# Tính KPI dựa trên bộ dữ liệu được gắn cờ (Flagged)

In [2]:
import pandas as pd


In [3]:
df = pd.read_csv("../../data/processed/Online_Retail_flagged.csv", encoding="ISO-8859-1", parse_dates=["InvoiceDate"])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,is_return,is_missing_customer,is_invalid_price,is_outlier_quantity,Revenue,YearMonth,Hour,Weekday
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,False,False,False,15.30,2010-12,8,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,False,20.34,2010-12,8,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,False,False,False,22.00,2010-12,8,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,False,20.34,2010-12,8,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,False,20.34,2010-12,8,Wednesday


In [4]:
return_late = df["is_return"].mean()
print(f'Return Rate: {return_late:.2%}')

Return Rate: 1.97%


In [5]:
cancelled_rate = df["is_cancelled"].mean()
print(f'Cancelled Rate: {cancelled_rate:.2%}')

Cancelled Rate: 1.72%


In [6]:
revenue_lost = df[df["is_return"]]["Revenue"].abs().sum()
print(f'Total Revenue Lost Due to Returns: ${revenue_lost:,.2f}')

Total Revenue Lost Due to Returns: $893,979.73


In [7]:
return_rate_product = (
    df.groupby("Description")["is_return"].mean()
    .sort_values(ascending=False)
    .head(10)
)
print(return_rate_product)

Description
wrongly coded-23343             1.0
water damage                    1.0
wrongly sold sets               1.0
wrongly sold as sets            1.0
wrongly marked. 23343 in box    1.0
water damaged                   1.0
thrown away-can't sell.         1.0
thrown away-can't sell          1.0
wet rusty                       1.0
thrown away                     1.0
Name: is_return, dtype: float64


## Tính kpi summary từ dữ liệu flagged

In [8]:
kpi_flagged = pd.DataFrame({
    "metric": [
        "Return Rate",
        "Cancel Rate",
        "Revenue Lost",
        "Return Impact"
    ],
    "value": [
        df["is_return"].mean(),
        df["is_cancelled"].mean(),
        df[df["is_return"]]["Revenue"].abs().sum(),
        df[df["is_return"]]["Revenue"].abs().sum() / df["Revenue"].sum()
    ]
})
kpi_flagged.to_csv("../../data/kpi/kpi_flagged_summary.csv", index=False)

In [9]:
net_revenue = df["Revenue"].sum()
print(f"Net Revenue: ${net_revenue:,.2f}")

Net Revenue: $9,726,006.95


In [10]:
gross_revenue = df[df["Quantity"] > 0]["Revenue"].sum()
lost_revenue = df[df["Quantity"] < 0]["Revenue"].abs().sum()

print(gross_revenue, lost_revenue)

10619986.68 893979.73


## Tính KPI hàng trả lại

In [11]:
return_analysis = df[df["is_return"]]

return_analysis_stats = pd.DataFrame({
    "metric": ["Total Returns", "Avg Return Value"],
    "value": [
        return_analysis.shape[0],
        return_analysis["Revenue"].abs().mean()
    ]
})

return_analysis_stats.to_csv("../../data/kpi/return_analysis.csv", index=False)

## Tính KPI hàng bị trống

In [12]:
lost_by_product = (
    df[df["is_return"]]
    .groupby("Description")["Revenue"]
    .apply(lambda x: x.abs().sum())
    .sort_values(ascending=False)
    .reset_index()
)

lost_by_product.to_csv("../../data/kpi/lost_by_product.csv", index=False)

## Tính KPI hàng trả lại theo thời gian

In [13]:
df["Month"] = df["InvoiceDate"].dt.to_period("M")

return_by_time = (
    df[df["is_return"]]
    .groupby("Month")["Revenue"]
    .apply(lambda x: x.abs().sum())
    .reset_index()
)

return_by_time.to_csv("../../data/kpi/return_by_time.csv", index=False)

## Tính KPI hàng bị hủy

In [14]:
cancel_analysis = df[df["is_cancelled"]]

cancel_stats = pd.DataFrame({
    "metric": ["Total Cancelled Orders"],
    "value": [cancel_analysis.shape[0]]
})

cancel_stats.to_csv("../../data/kpi/cancel_analysis.csv", index=False)

## Tính KPI số hàng bị trả lại theo tháng

In [15]:
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# ===== 3. KPI: Returned Quantity (loại bỏ cancel) =====
returned_quantity_by_month = (
    df[
        (df["Quantity"] < 0) & 
        (~df["InvoiceNo"].astype(str).str.startswith("C"))
    ]
    .groupby("YearMonth")["Quantity"]
    .sum()
    .reset_index()
    .sort_values("YearMonth")
)

# ===== 4. Convert sang số dương (dễ đọc) =====
returned_quantity_by_month["ReturnedQuantity"] = \
    returned_quantity_by_month["Quantity"].abs()

# ===== 5. Output =====
print("Returned Quantity by Month:\n", returned_quantity_by_month)

returned_quantity_by_month.to_csv(
    "../../data/kpi/returned_quantity_by_month.csv",
    index=False
)

Returned Quantity by Month:
    YearMonth  Quantity  ReturnedQuantity
0    2010-12     -4046              4046
1    2011-01     -8905              8905
2    2011-02     -5835              5835
3    2011-03    -27683             27683
4    2011-04     -9433              9433
5    2011-05    -13166             13166
6    2011-06    -45352             45352
7    2011-07    -10395             10395
8    2011-08     -6471              6471
9    2011-09    -18170             18170
10   2011-10    -33672             33672
11   2011-11    -18691             18691
12   2011-12     -5138              5138


## Tính KPI số hàng bán được theo tháng

In [16]:
positive_quantity_by_month = (
    df[
        (df["Quantity"] > 0) & 
        (~df["InvoiceNo"].astype(str).str.startswith("C"))
    ]
    .groupby("YearMonth")["Quantity"]
    .sum()
    .reset_index()
    .sort_values("YearMonth")
)

print("Positive Quantity by Month:\n", positive_quantity_by_month)

positive_quantity_by_month.to_csv(
    "../../data/kpi/positive_quantity_by_month.csv",
    index=False
)

Positive Quantity by Month:
    YearMonth  Quantity
0    2010-12    361094
1    2011-01    397030
2    2011-02    286074
3    2011-03    384023
4    2011-04    311314
5    2011-05    398686
6    2011-06    393633
7    2011-07    405473
8    2011-08    424266
9    2011-09    574169
10   2011-10    626373
11   2011-11    768468
12   2011-12    314416


In [17]:
date = "2011-06-14"

daily_quantity = df[
    df["InvoiceDate"].dt.date == pd.to_datetime(date).date()
]

positive_quantity = daily_quantity[daily_quantity["Quantity"] > 0]["Quantity"].sum()
negative_quantity = daily_quantity[daily_quantity["Quantity"] < 0]["Quantity"].sum()

print("Date:", date)
print("Positive Quantity:", positive_quantity)
print("Negative Quantity:", negative_quantity)
print("Returned Quantity (abs):", abs(negative_quantity))

Date: 2011-06-14
Positive Quantity: 14602
Negative Quantity: -28376
Returned Quantity (abs): 28376


In [24]:
all_returned_products = (
    df[
        (df["Quantity"] < 0) & 
        (df["InvoiceNo"].astype(str).str.startswith("C"))
    ]
    .groupby("Description")["Quantity"]
    .sum()
    .reset_index()
    .sort_values(by="Quantity")
)

print("Returned Products:\n", returned_products)

returned_products.to_csv(
    "../../data/kpi/all_returned_products.csv",
    index=False
)

Returned Products:
                               Description  Quantity
1184          PAPER CRAFT , LITTLE BIRDIE    -80995
1025       MEDIUM CERAMIC TOP STORAGE JAR    -74494
1459  ROTATING SILVER ANGELS T-LIGHT HLDR     -9376
1093                               Manual     -4066
592    FAIRY CAKE FLANNEL ASSORTED COLOUR     -3150
...                                   ...       ...
1552  SET OF 36 VINTAGE CHRISTMAS DOILIES        -1
1553        SET OF 4 DIAMOND NAPKIN RINGS        -1
1554       SET OF 4 ENGLISH ROSE COASTERS        -1
1472   RUSTIC WOODEN CABINET, GLASS DOORS        -1
1957              YELLOW/BLUE RETRO RADIO        -1

[1972 rows x 2 columns]


In [23]:
all_returned_by_customer = (
    df[
        (df["Quantity"] < 0) & 
        (df["InvoiceNo"].astype(str).str.startswith("C"))
    ]
    .groupby(["CustomerID", "Description"])["Quantity"]
    .sum()
    .reset_index()
    .sort_values(by="Quantity")
)

print("Returned by Customer:\n", returned_by_customer)

all_returned_by_customer.to_csv(
    "../../data/kpi/all_returned_by_customer.csv",
    index=False
)

Returned by Customer:
       CustomerID                          Description  Quantity
5888     16446.0          PAPER CRAFT , LITTLE BIRDIE    -80995
0        12346.0       MEDIUM CERAMIC TOP STORAGE JAR    -74215
5344     15838.0  ROTATING SILVER ANGELS T-LIGHT HLDR     -9360
5175     15749.0   FAIRY CAKE FLANNEL ASSORTED COLOUR     -3114
5176     15749.0          GIN + TONIC DIET METAL SIGN     -2000
...          ...                                  ...       ...
2679     14156.0           PICNIC BASKET WICKER SMALL        -1
2681     14156.0                REGENCY TEAPOT ROSES         -1
2684     14156.0   SET 6 SCHOOL MILK BOTTLES IN CRATE        -1
2687     14159.0                 DAIRY MAID TOASTRACK        -1
7806     18277.0             REGENCY CAKESTAND 3 TIER        -1

[7808 rows x 3 columns]
